Goal:
Transform raw features into a modeling-ready dataset by:
- Handling missing values
- Encoding categorical variables
- Engineering aggregate functions
- Preparing target variable for regression & uncertainity estimation

In [1]:
import numpy as np
import pandas as pd

In [2]:
df = pd.read_csv('C:/Users/Pratik/DS/house_prediction_uncertainty/data/raw/train.csv') 

In [3]:
df_2 = df.copy() # Create a copy of the original dataframe to work on feature engineering

In [4]:
# log transform the target variable
df_2['SalePrice_log'] = np.log1p(df_2['SalePrice'])

In [5]:
# Fill missing values in numerical columns with 0, since they likely indicate absence of that feature
num_cols = df_2.select_dtypes(include=['int64', 'float64']).columns
num_cols = num_cols.drop(['Id', 'SalePrice', 'SalePrice_log'])

df_2[num_cols] = df_2[num_cols].fillna(0)

In [6]:
df_2[num_cols].isna().sum() # check for any remaining missing values in numerical columns

MSSubClass       0
LotFrontage      0
LotArea          0
OverallQual      0
OverallCond      0
YearBuilt        0
YearRemodAdd     0
MasVnrArea       0
BsmtFinSF1       0
BsmtFinSF2       0
BsmtUnfSF        0
TotalBsmtSF      0
1stFlrSF         0
2ndFlrSF         0
LowQualFinSF     0
GrLivArea        0
BsmtFullBath     0
BsmtHalfBath     0
FullBath         0
HalfBath         0
BedroomAbvGr     0
KitchenAbvGr     0
TotRmsAbvGrd     0
Fireplaces       0
GarageYrBlt      0
GarageCars       0
GarageArea       0
WoodDeckSF       0
OpenPorchSF      0
EnclosedPorch    0
3SsnPorch        0
ScreenPorch      0
PoolArea         0
MiscVal          0
MoSold           0
YrSold           0
dtype: int64

In [7]:
# Fill missing values in categorical columns with 'None', indicating absence of that feature
cat_cols = df_2.select_dtypes(include=['object']).columns
df_2[cat_cols] = df_2[cat_cols].fillna('None')

In [8]:
df_2[cat_cols].isna().sum() # check for any remaining missing values in categorical columns

MSZoning         0
Street           0
Alley            0
LotShape         0
LandContour      0
Utilities        0
LotConfig        0
LandSlope        0
Neighborhood     0
Condition1       0
Condition2       0
BldgType         0
HouseStyle       0
RoofStyle        0
RoofMatl         0
Exterior1st      0
Exterior2nd      0
MasVnrType       0
ExterQual        0
ExterCond        0
Foundation       0
BsmtQual         0
BsmtCond         0
BsmtExposure     0
BsmtFinType1     0
BsmtFinType2     0
Heating          0
HeatingQC        0
CentralAir       0
Electrical       0
KitchenQual      0
Functional       0
FireplaceQu      0
GarageType       0
GarageFinish     0
GarageQual       0
GarageCond       0
PavedDrive       0
PoolQC           0
Fence            0
MiscFeature      0
SaleType         0
SaleCondition    0
dtype: int64

In [ ]:
# Categorical columns which are ordered , we will use ordinal encoding
quality_map = {
    'None': 0,
    'Ex': 5,
    'Gd': 4,
    'TA': 3,
    'Fa': 2,
    'Po': 1
}
ordinal_cols = [
    'ExterQual', 'ExterCond','KitchenQual','GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'FireplaceQu', 'HeatingQC', 'PoolQC'
]

# Map the quality ratings to numerical values for the ordinal columns
for col in ordinal_cols:
    if col in df_2.columns:
        df_2[col] = df_2[col].map(quality_map)

In [ ]:
# Label encoding for other categorical columns
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()

# Label encode the remaining categorical columns that are not ordinal
for col in cat_cols:
    if col not in ordinal_cols:
        df_2[col] = le.fit_transform(df_2[col])

In [11]:
# We have TotalBsmtSF , 1stFlrSF , 2ndFlrSF which are related to area of house , we can create a new feature TotalSF
df_2['TotalSF'] = df_2['TotalBsmtSF'] + df_2['1stFlrSF'] + df_2['2ndFlrSF']

In [12]:
# House Age feature
df_2['HouseAge'] = df_2['YrSold'] - df_2['YearBuilt']
df_2['RemodAge'] = df_2['YrSold'] - df_2['YearRemodAdd']

In [ ]:
# Creating TotalBathrooms feature by combining full and half bathrooms from both above ground and basement
df_2['TotalBathrooms'] = df_2['FullBath'] + (0.5 * df_2['HalfBath']) + df_2['BsmtFullBath'] + (0.5 * df_2['BsmtHalfBath'])

In [14]:
df_2.to_csv('C:/Users/Pratik/DS/house_prediction_uncertainty/data/processed/house_prices_features_v1.csv', index=False)

### Achieved the goals made at the start.

#### Ordinal Features:

- Quality based categorical features like KitchenQual were ordinal-encoded using a consistent numeric mapping reflecting their ordered nature, these capture the quality of structure & materials.

#### Nominal Features:

- High-cardinality nominal features were retained for tree-based models, which can naturally learn non0linear relations.

#### Numeric Feature Engineering:

- Created some features like TotalSF (Total Square Footage), TotalBathrooms to better represent overall size, also created houseage & remod age to better represent the property age & structural age. The target variable were log transformed to stabilize variance.
